<a href="https://colab.research.google.com/github/Shturman71/Colab/blob/Audio_test/Audio_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
from gtts import gTTS
from pydub import AudioSegment
import pandas as pd
import io # Необходим для StringIO при обработке TXT

# --- Пути к файлам ---
csv_file_path = 'qa_data.csv'
input_txt_file = 'qa_data.txt'

# --- Загрузка вопросов и ответов ---
qa_pairs = []

# Попытка загрузить из CSV сначала
if os.path.exists(csv_file_path):
    try:
        # Изменено: используем точку с запятой как разделитель И quotechar для обработки кавычек
        df = pd.read_csv(csv_file_path, sep=';', quotechar='"')

        # Normalize column names to lowercase
        df.columns = df.columns.str.lower()

        # Check if the dataframe has only one column, and if its name is 'question;answer'
        # This handles cases where `read_csv` failed to split the header by delimiter
        if len(df.columns) == 1 and df.columns[0] == 'question;answer':
            print("Обнаружен некорректный заголовок 'question;answer' как единый столбец. Попытка разделить.")
            # Split the single column into two, assuming the data is correctly delimited
            df_split = df[df.columns[0]].str.split(';', expand=True)
            if df_split.shape[1] == 2:
                df = df_split.rename(columns={0: 'question', 1: 'answer'})
            else:
                raise ValueError("Не удалось разделить единственный столбец 'question;answer' на два.")

        # Добавляем проверку на ожидаемые заголовки столбцов после всех попыток парсинга
        if 'question' not in df.columns or 'answer' not in df.columns:
            raise ValueError("CSV file does not have expected 'question' and 'answer' columns after parsing and normalization.")

        # Convert to dict, handling potential NaN values by converting them to empty strings
        qa_pairs = df.fillna('').to_dict('records')
        print(f"Успешно загружено {len(qa_pairs)} пар вопросов и ответов из '{csv_file_path}'.")
    except Exception as e:
        print(f"Ошибка при чтении CSV файла '{csv_file_path}': {e}. Попытка преобразовать из TXT.")
        # Fallback to TXT if CSV reading fails
else:
    print(f"Файл '{csv_file_path}' не найден. Попытка преобразовать из '{input_txt_file}'.")

# Если qa_pairs все еще пуст, значит, CSV не был загружен успешно или не найден,
# поэтому пытаемся конвертировать из TXT
if not qa_pairs:
    try:
        with open(input_txt_file, 'r', encoding='utf-8') as f:
            txt_content = f.read()

        # Используем pandas для чтения содержимого TXT как CSV, чтобы правильно обрабатывать кавычки
        # Изменено: используем точку с запятой как разделитель
        df_from_txt = pd.read_csv(io.StringIO(txt_content), sep=';', quotechar='"', encoding='utf-8')

        # Проверяем наличие ожидаемых столбцов
        if 'question' not in df_from_txt.columns or 'answer' not in df_from_txt.columns:
            raise ValueError("TXT file, when parsed as CSV, does not have expected 'question' and 'answer' columns.")

        # Сохраняем DataFrame в .csv файл (по умолчанию будет использовать запятую, если не указать sep)
        df_from_txt.to_csv(csv_file_path, index=False, encoding='utf-8')

        print(f"Файл '{input_txt_file}' успешно преобразован в '{csv_file_path}'.")
        print("Первые 5 строк нового CSV файла:")
        print(df_from_txt.head().to_string()) # Используем to_string() для вывода в консоль
        print(f"Всего вопросов и ответов в новом CSV файле: {len(df_from_txt)}")

        # Теперь загружаем только что созданные данные CSV в qa_pairs
        qa_pairs = df_from_txt.fillna('').to_dict('records') # Fill NaN with empty strings here too

    except FileNotFoundError:
        print(f"Ошибка: Файл '{input_txt_file}' не найден. Невозможно загрузить вопросы и ответы.")
    except Exception as e:
        print(f"Произошла ошибка при преобразовании TXT в CSV или загрузке: {e}")

# Если qa_pairs все еще пуст после всех попыток загрузки
if not qa_pairs:
    print("Список вопросов и ответов пуст, аудиофайл не будет создан.")
    # Выходим или предотвращаем дальнейшее выполнение, если данные не загружены
    # Текущий код продолжит выполнение, но блок 'if qa_pairs:' ниже его обработает.

# Язык по умолчанию для синтеза речи, если он не указан для конкретной фразы
default_speech_lang = 'en'

# === Настройки динамической паузы ===
base_pause_duration_ms = 750  # Базовая пауза (например, 0.75 секунды)
ms_per_word = 100             # Дополнительные миллисекунды за каждое слово
max_pause_duration_ms = 5000  # Максимальная продолжительность паузы

def calculate_dynamic_pause(text):
    # Ensure text is a string before splitting
    text = str(text)
    words = len(text.split())
    dynamic_pause = base_pause_duration_ms + (words * ms_per_word)
    return min(dynamic_pause, max_pause_duration_ms)

def generate_speech(text, lang=default_speech_lang, output_filename='temp_speech.mp3'):
    """Генерирует аудиофайл из текста с помощью gTTS."""
    try:
        tts = gTTS(text=text, lang=lang, slow=False);
        tts.save(output_filename);
        return AudioSegment.from_mp3(output_filename);
    except Exception as e:
        print(f"Ошибка при генерации речи для '{text}' (язык: {lang}): {e}");
        return AudioSegment.silent(duration=100); # Возвращаем короткую тишину в случае ошибки
    finally:
        if os.path.exists(output_filename):
            os.remove(output_filename);

def create_audio_with_pauses(qa_list):
    """Создает общий аудиофайл из вопросов, ответов и пауз."""
    combined_audio = AudioSegment.empty();

    for i, qa in enumerate(qa_list):
        # Ensure question_text and answer_text are strings
        question_text = str(qa['question']);
        answer_text = str(qa['answer']);

        # Определяем язык для текущей фразы, используя 'lang' из qa_pairs, или язык по умолчанию
        current_lang = qa.get('lang', default_speech_lang)

        # Рассчитываем динамическую паузу для текущего вопроса
        pause_after_question_ms = calculate_dynamic_pause(question_text)
        silence_after_question = AudioSegment.silent(duration=pause_after_question_ms)

        print(f"Обработка: Вопрос {i+1} - '{question_text}' (язык: {current_lang})");
        question_audio = generate_speech(question_text, lang=current_lang, output_filename=f"question_{i}.mp3");
        combined_audio += question_audio;

        print(f"Добавление паузы ({pause_after_question_ms} мс) после вопроса {i+1}");
        combined_audio += silence_after_question;

        print(f"Обработка: Ответ {i+1} - '{answer_text}' (язык: {current_lang})");
        answer_audio = generate_speech(answer_text, lang=current_lang, output_filename=f"answer_{i}.mp3");
        combined_audio += answer_audio;

        if i < len(qa_list) - 1:
            # Рассчитываем динамическую паузу для текущего ответа
            pause_after_answer_ms = calculate_dynamic_pause(answer_text)
            silence_after_answer = AudioSegment.silent(duration=pause_after_answer_ms)
            print(f"Добавление паузы ({pause_after_answer_ms} мс) после ответа {i+1}");
            combined_audio += silence_after_answer;

    return combined_audio;

# Генерируем аудио только если qa_pairs не пуст
if qa_pairs:
    # Передаем только qa_pairs, так как pause_duration_ms больше не фиксирован
    final_audio = create_audio_with_pauses(qa_pairs);

    # Сохраняем итоговый аудиофайл
    output_audio_path = "q_and_a_audio.mp3";
    try:
        final_audio.export(output_audio_path, format="mp3");
        print(f"Аудиофайл успешно создан: {output_audio_path}");
    except Exception as e:
        print(f"Ошибка при сохранении аудиофайла '{output_audio_path}': {e}");
else:
    print("Список вопросов и ответов пуст, аудиофайл не будет создан.")

Обнаружен некорректный заголовок 'question;answer' как единый столбец. Попытка разделить.
Успешно загружено 31 пар вопросов и ответов из 'qa_data.csv'.
Обработка: Вопрос 1 - 'What seemed to be your fault when you were young?' (язык: en)
Добавление паузы (1750 мс) после вопроса 1
Обработка: Ответ 1 - '"When I was young, everything that went wrong in my house seemed to be my fault."' (язык: en)
Добавление паузы (2350 мс) после ответа 1
Обработка: Вопрос 2 - 'What did your brothers try to do once?' (язык: en)
Добавление паузы (1550 мс) после вопроса 2
Обработка: Ответ 2 - '"Once, my brothers tried to make cookies."' (язык: en)
Добавление паузы (1450 мс) после ответа 2
Обработка: Вопрос 3 - 'What did they blend together and what did they make?' (язык: en)
Добавление паузы (1750 мс) после вопроса 3
Обработка: Ответ 3 - '"They blended flour and ginger and made a disgusting paste."' (язык: en)
Добавление паузы (1750 мс) после ответа 3
Обработка: Вопрос 4 - 'What happened when they tried to wa

In [11]:
from IPython.display import Audio

output_audio_path = "q_and_a_audio.mp3"
if os.path.exists(output_audio_path):
    display(Audio(output_audio_path, autoplay=False))
else:
    print(f"Аудиофайл '{output_audio_path}' не найден.")

In [2]:
input_txt_file = 'qa_data.txt'

try:
    with open(input_txt_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        if len(lines) >= 8:
            print(f"Содержимое 8-й строки файла '{input_txt_file}':\n{lines[7].strip()}") # 8th line is index 7
        else:
            print(f"Файл '{input_txt_file}' содержит менее 8 строк.")
except FileNotFoundError:
    print(f"Ошибка: Файл '{input_txt_file}' не найден.")
except Exception as e:
    print(f"Произошла ошибка при чтении файла: {e}")

Ошибка: Файл 'qa_data.txt' не найден.


In [3]:
import pandas as pd
import io

# Укажите путь к вашему .txt файлу
input_txt_file = 'qa_data.txt'
output_csv_file = 'qa_data.csv'

try:
    # Читаем .txt файл, чтобы взять все строки
    with open(input_txt_file, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()

    processed_data = []
    # Предполагаем, что первая строка — это заголовок
    header = all_lines[0].strip().split(',')
    if len(header) == 2:
        processed_data.append(header)
    else:
        print(f"Ошибка: Заголовок файла '{input_txt_file}' не содержит 2 поля: {all_lines[0].strip()}")
        raise ValueError("Некорректный формат заголовка")

    for i, line in enumerate(all_lines[1:]): # Начинаем со второй строки (индекс 1)
        line_number = i + 2 # Фактический номер строки в файле
        parts = line.strip().split(',', 1) # Разделяем только по первой запятой
        if len(parts) == 2:
            processed_data.append(parts)
        else:
            print(f"Пропускается проблемная строка {line_number} из-за неправильного формата: '{line.strip()}'")

    # Конвертируем в DataFrame, пропуская первую строку, если она является заголовком, который уже обработан
    if len(processed_data) > 1:
        df_from_txt = pd.DataFrame(processed_data[1:], columns=processed_data[0])
    else:
        df_from_txt = pd.DataFrame(columns=processed_data[0]) # Пустой DataFrame с заголовком

    # Сохраняем DataFrame в .csv файл
    # index=False предотвращает запись индекса DataFrame в CSV файл
    df_from_txt.to_csv(output_csv_file, index=False, encoding='utf-8')

    print(f"Файл '{input_txt_file}' успешно преобразован в '{output_csv_file}'.")
    print("Первые 5 строк нового CSV файла:")
    display(df_from_txt.head())
    print(f"Всего вопросов и ответов в новом CSV файле: {len(df_from_txt)}")

except FileNotFoundError:
    print(f"Ошибка: Файл '{input_txt_file}' не найден. Убедитесь, что файл существует и путь к нему указан верно.")
except Exception as e:
    print(f"Произошла ошибка при преобразовании файла: {e}")

Ошибка: Файл 'qa_data.txt' не найден. Убедитесь, что файл существует и путь к нему указан верно.


Теперь, когда код готов, вам нужно создать файл `qa_data.csv` в корневой директории вашего Colab-проекта. Файл должен иметь две колонки: `question` и `answer`. Вот пример содержимого такого файла:

```csv
question,answer
Какой ваш любимый цвет?,Мой любимый цвет — синий.
Как вас зовут?,Я — большая языковая модель Google.
Какова столица Франции?,Столица Франции — Париж.
```

Вы можете загрузить этот файл, используя панель слева (иконка папки), или создать его программно.

In [4]:
!pip install gtts pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
